# Phase 1: Data Cleaning and Preprocessing

**Course:** Foundations of Data Science (23CSE351) · Group 3  
**Owner:** Person A  
**Dataset:** DataCo Smart Supply Chain

This notebook documents Phase 1 end to end. It loads the raw CSV, cleans it under the project rules, writes the shared cleaned file, and runs the independent validator.

### Absolute rules used here
- Never delete rows (negative-profit rows are the loss class)
- Never scale or standardise (teammates scale after their own train/test split)
- Never create a train/test split in this phase
- Never derive features from the shipping date (late-delivery leakage)
- Never modify the raw CSV
- Keep surviving original column names


## 0. Setup

Paths are relative to the repository root. Outputs go under `A_preprocessing_eda/outputs/`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

cwd = Path.cwd().resolve()
if (cwd / "dataset" / "DataCoSupplyChainDataset.csv").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "dataset" / "DataCoSupplyChainDataset.csv").exists():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find dataset/DataCoSupplyChainDataset.csv. "
        "Open the notebook from the repository root or A_preprocessing_eda/."
    )

RAW_PATH = REPO_ROOT / "dataset" / "DataCoSupplyChainDataset.csv"
OUTPUT_DIR = REPO_ROOT / "A_preprocessing_eda" / "outputs"
PLOT_DIR = OUTPUT_DIR / "outlier_plots"
CLEANED_PATH = OUTPUT_DIR / "cleaned_supply_chain.csv"
VALIDATOR = REPO_ROOT / "A_preprocessing_eda" / "phase1_validate.py"

RANDOM_STATE = 42
TOP_N = 10
FLOAT_TOL = 1e-9
SKEW_ABS_THRESHOLD = 1.0
ORDER_DATE_COL = "order date (DateOrders)"
SHIP_DATE_COL = "shipping date (DateOrders)"
ORDER_DATE_FORMAT = "%m/%d/%Y %H:%M"
ISO_FORMAT = "%Y-%m-%d %H:%M:%S"
WEEKDAY_NAMES = (
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday",
)

np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RAW_PATH :", RAW_PATH)
print("EXISTS   :", RAW_PATH.is_file())


## 1. Load and inspect

Load with `encoding="latin-1"`. Expected raw shape: 180,519 × 53.


In [ ]:
df = pd.read_csv(RAW_PATH, encoding="latin-1", low_memory=False)
raw = df.copy()
n_rows, n_cols = df.shape
print(f"Shape: {n_rows:,} rows × {n_cols} columns")
print(f"Duplicate rows: {int(df.duplicated().sum()):,}")
print(f"Memory (deep): {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
display(df.dtypes.to_frame("dtype"))

nulls = df.isna().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print(f"Columns with nulls: {len(nulls)} of {n_cols}")
display(pd.DataFrame({
    "null_count": nulls,
    "null_pct": (100 * nulls / n_rows).round(4),
}))

late_pct = 100 * (df["Late_delivery_risk"] == 1).mean()
loss_pct = 100 * (df["Benefit per order"] < 0).mean()
zero_pct = 100 * (df["Benefit per order"] == 0).mean()
print(f"Late_delivery_risk = 1: {late_pct:.4f}%")
print(f"Benefit per order < 0 : {loss_pct:.4f}%")
print(f"Benefit per order = 0 : {zero_pct:.4f}%")


## 2. Drop columns

Explicit lists only: empty/useless, personal, identifiers, late-delivery leakage, and profit leakage.

`Benefit per order` is kept (OLS target and source of `loss`).  
`Days for shipment (scheduled)` is kept (known at order time).


In [ ]:
EMPTY = [
    "Product Description", "Order Zipcode", "Product Status", "Product Image",
]
PERSONAL = [
    "Customer Password", "Customer Email", "Customer Fname",
    "Customer Lname", "Customer Street",
]
IDS = [
    "Customer Id", "Order Customer Id", "Order Id", "Order Item Id",
    "Category Id", "Department Id", "Product Card Id",
    "Product Category Id", "Order Item Cardprod Id",
]
LATE_LEAK = [
    "Delivery Status", "Days for shipping (real)", SHIP_DATE_COL,
]
PROFIT_LEAK = [
    "Order Profit Per Order", "Order Item Profit Ratio",
]

diff_benefit = int(
    ((df["Benefit per order"] - df["Order Profit Per Order"]).abs() > FLOAT_TOL).sum()
)
print(f"Benefit per order vs Order Profit Per Order differing rows: {diff_benefit}")

to_drop = EMPTY + PERSONAL + IDS + LATE_LEAK + PROFIT_LEAK
missing = [c for c in to_drop if c not in df.columns]
assert not missing, f"Listed columns missing: {missing}"

df = df.drop(columns=to_drop)
assert len(df) == n_rows
print(f"After Step 2 drops: {df.shape[1]} columns, {len(df):,} rows")
assert SHIP_DATE_COL not in df.columns
assert "Days for shipment (scheduled)" in df.columns
assert "Benefit per order" in df.columns


## 3. Parse the order date

Derive year, month, and weekday. Do **not** use the shipping date.


In [ ]:
parsed = pd.to_datetime(df[ORDER_DATE_COL], format=ORDER_DATE_FORMAT, errors="coerce")
failed = int(parsed.isna().sum())
if failed:
    raise ValueError(f"{failed} order dates failed to parse with {ORDER_DATE_FORMAT}")

df[ORDER_DATE_COL] = parsed
df["order_year"] = parsed.dt.year.astype("int64")
df["order_month"] = parsed.dt.month.astype("int64")
df["order_weekday"] = parsed.dt.dayofweek.astype("int64")  # Monday = 0
df["order_weekday_name"] = df["order_weekday"].map(dict(enumerate(WEEKDAY_NAMES)))

print("Order date min:", df[ORDER_DATE_COL].min())
print("Order date max:", df[ORDER_DATE_COL].max())
print("Columns after dates:", df.shape[1])


## 4. Near-duplicates and remaining nulls

Keep `Order Item Total` over `Sales per customer` when identical. Drop other exact numeric duplicates. Fill `Customer Zipcode` nulls with `"Unknown"`.


In [ ]:
def differing_rows(a, b, tol=FLOAT_TOL):
    left = pd.to_numeric(a, errors="coerce")
    right = pd.to_numeric(b, errors="coerce")
    mismatch = left.isna() ^ right.isna()
    gap = (left - right).abs()
    value_mismatch = gap.notna() & (gap > tol)
    return int(value_mismatch.sum() + mismatch.sum()), float(gap.max() if gap.notna().any() else 0.0)

if "Sales per customer" in df.columns and "Order Item Total" in df.columns:
    n_diff, max_abs = differing_rows(df["Sales per customer"], df["Order Item Total"])
    print(f"Sales per customer vs Order Item Total: differ={n_diff}, max_abs={max_abs}")
    if n_diff == 0:
        df = df.drop(columns=["Sales per customer"])
        print("Dropped Sales per customer (exact duplicate of Order Item Total)")

protected = {
    "Benefit per order", "Late_delivery_risk", "Sales", "Order Item Quantity",
    "Days for shipment (scheduled)", "Order Item Total", "Order Item Product Price",
    "order_year", "order_month", "order_weekday",
}
numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
drop_exact = []
for i, c1 in enumerate(numeric):
    for c2 in numeric[i + 1:]:
        n_diff, max_abs = differing_rows(df[c1], df[c2])
        if n_diff == 0:
            if c2 not in protected:
                drop_exact.append(c2)
                print(f"Exact duplicate: drop {c2!r}, keep {c1!r}")
            elif c1 not in protected:
                drop_exact.append(c1)
                print(f"Exact duplicate: drop {c1!r}, keep {c2!r}")
            else:
                print(f"Exact duplicate but both protected: {c1!r}, {c2!r}")
drop_exact = list(dict.fromkeys(drop_exact))
if drop_exact:
    df = df.drop(columns=drop_exact)

numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
corr = df[numeric].corr(method="pearson")
print("High-correlation pairs (|r| > 0.95):")
for i, a in enumerate(numeric):
    for b in numeric[i + 1:]:
        r = corr.loc[a, b]
        if pd.notna(r) and abs(r) > 0.95:
            print(f"  {a} | {b} | r={r:.6f}")

if "Customer Zipcode" in df.columns:
    n_zip = int(df["Customer Zipcode"].isna().sum())
    non_null = df["Customer Zipcode"].dropna()
    if len(non_null) and not np.all(np.equal(np.mod(non_null.to_numpy(dtype=float), 1), 0)):
        raise ValueError("Customer Zipcode has non-integral values")
    as_int = df["Customer Zipcode"].round(0).astype("Int64").astype(str)
    df["Customer Zipcode"] = as_int.mask(df["Customer Zipcode"].isna(), "Unknown")
    print(f"Customer Zipcode: filled {n_zip} nulls with 'Unknown'")

assert int(df.isna().sum().sum()) == 0, "Nulls remain after Step 4"
print(f"After Step 4: {df.shape[1]} columns, 0 nulls")


## 5. Outlier review (document only)

IQR fences for `Sales`, `Benefit per order`, and `Order Item Quantity`. No rows are deleted or capped. Skew transforms are added as **new** columns.


In [ ]:
def iqr_report(series):
    q1 = float(series.quantile(0.25))
    q3 = float(series.quantile(0.75))
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outside = (series < lower) | (series > upper)
    return {
        "Q1": q1, "Q3": q3, "IQR": iqr,
        "lower": lower, "upper": upper,
        "n_out": int(outside.sum()),
        "pct_out": 100 * float(outside.mean()),
        "skew": float(series.skew()),
        "n_neg_out": int((outside & (series < 0)).sum()),
    }

reports = {c: iqr_report(df[c]) for c in ["Sales", "Benefit per order", "Order Item Quantity"]}
display(pd.DataFrame(reports).T)

assert (df["Sales"] >= 0).all(), "Sales must be non-negative for log1p"
df["Sales_log"] = np.log1p(df["Sales"].to_numpy(dtype=float))
benefit = df["Benefit per order"].to_numpy(dtype=float)
df["Benefit_signed_log"] = np.sign(benefit) * np.log1p(np.abs(benefit))

qty_skew = float(df["Order Item Quantity"].skew())
print(f"Order Item Quantity skewness: {qty_skew:.6f}")
if abs(qty_skew) > SKEW_ABS_THRESHOLD:
    df["Order_Item_Quantity_log"] = np.log1p(df["Order Item Quantity"].to_numpy(dtype=float))
    print("Added Order_Item_Quantity_log")
else:
    print(f"No quantity log column (threshold |skew| > {SKEW_ABS_THRESHOLD})")

print("Sales skew before/after:", reports["Sales"]["skew"], float(df["Sales_log"].skew()))
print(
    "Benefit skew before/after:",
    reports["Benefit per order"]["skew"],
    float(df["Benefit_signed_log"].skew()),
)

for old in PLOT_DIR.glob("*.png"):
    old.unlink()

def save_box(series, title, ylabel, path):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(y=series, ax=ax, color="#1f4e79", fliersize=2, linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    fig.tight_layout()
    fig.savefig(path, dpi=150, facecolor="white")
    plt.close(fig)

def save_hist(series, title, xlabel, path):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(x=series, bins=40, ax=ax, color="#1f4e79", edgecolor="white", linewidth=0.3)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Orders")
    fig.tight_layout()
    fig.savefig(path, dpi=150, facecolor="white")
    plt.close(fig)

save_box(df["Sales"], "Sales: box plot (1.5 IQR fences)", "Sales", PLOT_DIR / "box_Sales.png")
save_hist(df["Sales"], "Sales before transform", "Sales", PLOT_DIR / "hist_Sales_before.png")
save_hist(df["Sales_log"], "Sales after transform", "Sales_log", PLOT_DIR / "hist_Sales_after.png")

save_box(
    df["Benefit per order"],
    "Benefit per order: box plot (1.5 IQR fences)",
    "Benefit per order",
    PLOT_DIR / "box_Benefit_per_order.png",
)
save_hist(
    df["Benefit per order"],
    "Benefit per order before transform",
    "Benefit per order",
    PLOT_DIR / "hist_Benefit_per_order_before.png",
)
save_hist(
    df["Benefit_signed_log"],
    "Benefit per order after transform",
    "Benefit_signed_log",
    PLOT_DIR / "hist_Benefit_per_order_after.png",
)
save_box(
    df["Order Item Quantity"],
    "Order Item Quantity: box plot (1.5 IQR fences)",
    "Order Item Quantity",
    PLOT_DIR / "box_Order_Item_Quantity.png",
)
print("Plots written to", PLOT_DIR.relative_to(REPO_ROOT))


## 6. Encode categoricals

Keep original text columns **and** add integer 0/1 dummies (`drop_first=False`).  
High-cardinality columns use top-10 grouping (ties broken alphabetically), then one-hot.


In [ ]:
LOW_CARD = [
    "Shipping Mode", "Market", "Customer Segment", "Type", "Department Name", "Order Region",
]
HIGH_CARD = ["Category Name", "Order State", "Order Country"]

def one_hot(series, prefix):
    text = series.astype(str)
    levels = sorted(text.unique().tolist())
    dummies = pd.get_dummies(text, prefix=prefix, prefix_sep="_", dtype=int)
    expected = [f"{prefix}_{level}" for level in levels]
    dummies = dummies.loc[:, expected]
    assert dummies.sum(axis=1).eq(1).all()
    assert set(np.unique(dummies.to_numpy())).issubset({0, 1})
    return dummies

dummy_frames = []
for col in LOW_CARD:
    dummies = one_hot(df[col], col)
    dummy_frames.append(dummies)
    print(f"{col}: {dummies.shape[1]} levels one-hot encoded")

grouped_cols = []
for col in HIGH_CARD:
    counts = df[col].astype(str).value_counts()
    ranked = sorted(((str(k), int(v)) for k, v in counts.items()), key=lambda kv: (-kv[1], kv[0]))
    kept = [label for label, _ in ranked[:TOP_N]]
    keep_set = set(kept)
    text = df[col].astype(str)
    grouped = text.where(text.isin(keep_set), other="Other") if len(ranked) > TOP_N else text
    gname = f"{col}_grouped"
    df[gname] = grouped
    grouped_cols.append(gname)
    covered = int(text.isin(keep_set).sum()) if len(ranked) > TOP_N else n_rows
    print(f"{col}: {len(ranked)} unique → top {min(TOP_N, len(ranked))} cover {100 * covered / n_rows:.4f}%")
    dummies = one_hot(df[gname], gname)
    dummy_frames.append(dummies)

df = pd.concat([df, *dummy_frames], axis=1)
print("Columns after encoding:", df.shape[1])


## 7. Add targets

- `late` = `Late_delivery_risk`
- `loss` = 1 if `Benefit per order` < 0 else 0


In [ ]:
df["late"] = df["Late_delivery_risk"].astype("int64")
df["loss"] = (df["Benefit per order"] < 0).astype("int64")

assert set(df["late"].unique()).issubset({0, 1})
assert set(df["loss"].unique()).issubset({0, 1})
assert (df["late"] == df["Late_delivery_risk"]).all()
assert (df["loss"] == (df["Benefit per order"] < 0).astype("int64")).all()

print("late class counts:")
print(df["late"].value_counts().sort_index())
print("loss class counts:")
print(df["loss"].value_counts().sort_index())
zero = int((df["Benefit per order"] == 0).sum())
print(f"Zero-profit rows: {zero:,} ({100 * zero / n_rows:.4f}%) — all coded as not loss")


## 8. Column order, save, and validate

Original columns first, then derived dates, transforms, grouped text, one-hots, then `late` and `loss`.


In [ ]:
original_order = [c for c in raw.columns if c in df.columns]
derived_dates = ["order_year", "order_month", "order_weekday", "order_weekday_name"]
transforms = [c for c in ["Sales_log", "Benefit_signed_log", "Order_Item_Quantity_log"] if c in df.columns]
prefixes = LOW_CARD + [f"{x}_grouped" for x in HIGH_CARD]
one_hot_cols = [
    c for c in df.columns
    if any(c.startswith(p + "_") for p in prefixes)
    and c not in original_order
    and c not in grouped_cols
]

ordered = original_order + derived_dates + transforms + grouped_cols + one_hot_cols + ["late", "loss"]
seen = set()
ordered = [c for c in ordered if not (c in seen or seen.add(c))]
missing = [c for c in df.columns if c not in ordered]
assert not missing, f"Unplaced columns: {missing}"
df = df.loc[:, ordered]

for col in ["Sales", "Benefit per order", "Order Item Quantity"]:
    assert np.array_equal(df[col].to_numpy(), raw[col].to_numpy()), f"{col} was altered"

export = df.copy()
export[ORDER_DATE_COL] = export[ORDER_DATE_COL].dt.strftime(ISO_FORMAT)
export.to_csv(CLEANED_PATH, index=False, encoding="utf-8", lineterminator="\n")

size_mb = CLEANED_PATH.stat().st_size / (1024 ** 2)
print(f"Wrote {CLEANED_PATH.relative_to(REPO_ROOT)} ({size_mb:.2f} MB)")
print(f"Final shape: {df.shape[0]:,} × {df.shape[1]}")

import subprocess

result = subprocess.run(
    [
        sys.executable,
        str(VALIDATOR),
        "--raw",
        str(RAW_PATH.relative_to(REPO_ROOT)),
        "--cleaned",
        str(CLEANED_PATH.relative_to(REPO_ROOT)),
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Validator failed with exit code {result.returncode}")
print("Phase 1 notebook run complete.")


## Hand-off notes for teammates

1. Use `A_preprocessing_eda/outputs/cleaned_supply_chain.csv`.
2. The file is **unscaled** and has **no train/test split**.
3. Targets: `late` / `Late_delivery_risk`, `loss`, and `Benefit per order` (OLS only).
4. Drop one dummy level per one-hot group before intercept models.
5. Do not use `Order Status` until the group decides (possible post-order leakage).
6. Full report: `A_preprocessing_eda/outputs/PHASE1_REPORT.md`  
   Cleaning log: `A_preprocessing_eda/outputs/cleaning_log.md`
